# 학생 모델 QLoRA 학습

**실행 전 준비물** (Google Drive `MyDrive/ade-project/`에 업로드):
- `unified.jsonl` (항상 필요)
- `teacher_outputs.jsonl` (R2/R3/R4 조건에만 필요 — generate_teacher_data.ipynb의 출력)
- `cadec_v1_sct.jsonl` (R1 조건에만 필요 — 로컬 `research-project/data/cadec_v1_sct.jsonl`)

`fold_assignment.csv`는 git에 커밋되어 있어 clone 시 자동으로 딸려온다.

**주의**: R2/R3/R4 생성(generate_teacher_data.ipynb)과 같은 세션에서 돌리지 말 것 (VRAM 부족).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/ade-project'

In [ ]:
REPO_URL = 'https://github.com/Gaeul5/Oracle_healthcare-bio_sLLM.git'

import os
if not os.path.exists('/content/repo'):
    !git clone -q {REPO_URL} /content/repo
else:
    !git -C /content/repo pull -q
%cd /content/repo/research-project

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes peft datasets

## 실험 설정

이 셀만 바꿔서 조건/모델 크기/축을 조합해 재실행하면 된다 (30개 조합을 셀마다 따로 만드는 대신 여기 값만 바꿔가며 반복 실행).

- `CONDITION`: `R0` / `R1` / `R2` / `R3` / `R4` (R1은 CADEC v1만, R2-R4는 teacher_outputs.jsonl 필요)
- 주 축을 쓰려면 `TRAIN_DOMAIN`을 `'forum'` 또는 `'literature'`로, `CV_ROUND`는 `None`
- 보조 축(포럼 내부 4-fold)을 쓰려면 `TRAIN_DOMAIN=None`, `CV_ROUND`를 1~4로 (R1은 1 또는 4만 가능 — cadec_v1이 fold 2라 2,3은 학습 예시 0건)

In [ ]:
MODEL_SIZE = '0.5b'        # '0.5b' | '1.5b' | '3b'
CONDITION = 'R3'           # 'R0' | 'R1' | 'R2' | 'R3' | 'R4'
TRAIN_DOMAIN = 'forum'     # 'forum' | 'literature' | None
CV_ROUND = None            # 1 | 2 | 3 | 4 | None
SEED = 0

assert (TRAIN_DOMAIN is None) != (CV_ROUND is None), 'TRAIN_DOMAIN과 CV_ROUND 중 정확히 하나만 지정'

axis_tag = TRAIN_DOMAIN if TRAIN_DOMAIN else f'cvround{CV_ROUND}'
run_name = f'{MODEL_SIZE}_{CONDITION}_{axis_tag}_seed{SEED}'

cmd = (
    f'python src/train_qlora.py '
    f'--unified {DRIVE_DIR}/unified.jsonl '
    f'--teacher-outputs {DRIVE_DIR}/teacher_outputs.jsonl '
    f'--cadec-v1-sct {DRIVE_DIR}/cadec_v1_sct.jsonl '
    f'--fold-assignment data/fold_assignment.csv '
    f'--model-size {MODEL_SIZE} --condition {CONDITION} '
    + (f'--train-domain {TRAIN_DOMAIN} ' if TRAIN_DOMAIN else f'--cv-round {CV_ROUND} ')
    + f'--seed {SEED} '
    f'--out-dir {DRIVE_DIR}/adapters/{run_name}'
)
print(cmd)

In [ ]:
!{cmd}